# WS4 — GRPO Training (Path A: Unsloth + TRL)

FarmSimulation hackathon, Person B. Trains `Qwen2.5-0.5B-Instruct` on Task 1 via TRL `GRPOTrainer` with Unsloth's vLLM-backed fast inference and 4-bit + LoRA.

**Status (H+1):** Cells 1–5 ✅ green on Colab T4 (model loads, LoRA attaches). Cells 6, 8, 9, 10 scaffolded — independent of WS2/WS3. §20.7's cells 5 (FarmEnvClient) and 7 (5 reward functions) are pending HANDOFF #2 (A's WS2 merge — economy hardening). Cells 11–13 (plots, Hub upload, metadata) are post-training.

Cell numbering follows `IMPLEMENTATION_PLAN.md` §20.7. Notebook indices: header(0) · install(1) · version-check(2) · login(3) · model(4) · LoRA(5) · cell-6(6) · cell-8(7) · cell-9(8) · cell-10(9). §20.7's cells 5 and 7 are deliberately absent — they will be inserted after HANDOFF #2.

**Runtime:** Colab T4 (free tier OK for cells 1–5 smoke test; cells 7–9 are CPU-only scaffolding). Final 50-iter GRPO training will run on the dedicated L4 Space at H+18-19.

**Critical flags (do not change without re-reading §20.3):**
- `fast_inference=True` requires `load_in_4bit=True` (vLLM backend, ~10× generation speedup).
- `use_gradient_checkpointing="unsloth"` — **string**, not `True`. Unsloth's custom impl saves an extra ~30% VRAM.
- `gpu_memory_utilization=0.7` — drop to `0.5` only if you OOM during rollout.
- `enforce_eager=True` — disables vLLM CUDA graph compilation; ~10% slower generation but no graph-capture OOM on T4.
- `lora_dropout=0.05` per §20.3 — accepts the Unsloth fast-path penalty (~30-50% slower) for regularization. Revisit at H+18 before L4 run.

**Pin matrix (empirically validated on Colab T4 H+1):** `unsloth 2026.4.8` · `unsloth_zoo 2026.4.9` · `trl 0.22.2` · `transformers 4.56.2` · `huggingface_hub 0.34.0` · `vllm 0.19.1` · `torch 2.10.0+cu128`.

In [ ]:
!pip install -q unsloth vllm \
trl==0.22.2 \
transformers==4.56.2 \
huggingface_hub==0.34.0 \
openenv-core wandb

In [ ]:
import importlib.metadata as _md

for _pkg in ["unsloth", "unsloth_zoo", "trl", "transformers", "torch", "vllm", "huggingface_hub"]:
    try:
        print(f"{_pkg:20} = {_md.version(_pkg)}")
    except:
        print(f"{_pkg:20} = NOT INSTALLED")

In [ ]:
from huggingface_hub import login as hf_login
hf_login()

import wandb
wandb.login()

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length=1104,                # = max_prompt_length (1024) + max_completion_length (80)
    load_in_4bit=True,                  # 4-bit base, LoRA stays bf16
    fast_inference=True,                # vLLM backend → ~10× faster generation
    max_lora_rank=16,
    gpu_memory_utilization=0.7,         # leave 30% for activations + KV cache
    enforce_eager=True,                 # disable CUDA graph compilation — T4 stability
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,                      # alpha = 2 × r is Unsloth's default rule
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",   # NOT True/False — the string "unsloth" enables their custom impl
    random_state=3407,
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6 (per §20.7) — parse_action() helper
# Lifted verbatim from inference.py:142-155. Pure JSON parsing with regex
# fallback and a "wait" default. Used by reward functions (cell 7) and
# by FarmEnvClient.step() (cell 5) to translate model output → env action.
# ─────────────────────────────────────────────────────────────────────────────
import json
import re
from typing import Any, Dict

FALLBACK_ACTION: Dict[str, Any] = {"action_type": "wait"}
_JSON_OBJECT_RE = re.compile(r"\{[^{}]*\}", re.DOTALL)


def parse_action(response_text: str) -> Dict[str, Any]:
    """Extract a JSON action dict from raw LLM output. Returns FALLBACK_ACTION on any failure."""
    if not response_text or not response_text.strip():
        return dict(FALLBACK_ACTION)
    try:
        return json.loads(response_text.strip())
    except json.JSONDecodeError:
        pass
    for match in _JSON_OBJECT_RE.finditer(response_text):
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            continue
    return dict(FALLBACK_ACTION)


# Smoke test
assert parse_action('{"action_type":"plant","plot_id":0,"seed_type":"wheat"}')["action_type"] == "plant"
assert parse_action("garbage text {bad json")["action_type"] == "wait"
assert parse_action("Sure, here is the action: {\"action_type\":\"sell\",\"seed_type\":\"corn\",\"quantity\":5}")["seed_type"] == "corn"
print("parse_action ✅")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8 (per §20.7) — prompt_dataset
# Builds NUM_PROMPTS rows of (chat-formatted prompt, noise_seed, task_id).
# Each row is one GRPO training example — TRL samples K=4 completions per row.
#
# Format: applies the tokenizer's chat template (Qwen2.5 ChatML) per §20.6
# pitfall #7. Inside the user message, we use A's narrative format markers
# (### Observation: / ### Thought:) which the model can learn to anchor to.
#
# TODO[WS2]: replace _synthetic_narrative_text() with real env.reset()
# observations once cell 5 (FarmEnvClient) lands. The current synthetic text
# is enough to test prompt-template correctness and dataset shape.
# ─────────────────────────────────────────────────────────────────────────────
import os
from datasets import Dataset

NUM_PROMPTS = 200
SPACE_URL = os.getenv("FARMING_ENV_URL", "TBD")   # set on Colab once A posts the URL

SYSTEM_PROMPT = """You are an expert farmer managing 4 land plots across rotating climates. Each turn, observe the farm's narrative state and respond with a single JSON action.

Action types (12): wait | end_day | buy_seeds | plant | irrigate | harvest | sell | pump_water | apply_fertilizer | spray_pesticide | pull_weeds | buy_plot

Each action's required fields vary (e.g. plant needs plot_id + seed_type; sell needs seed_type + quantity). You have 10 labor hours per day; most actions cost 0.5–4.0 hours, wait costs 1.0, end_day costs 0. When labor runs out the day auto-advances.

Respond with ONLY a JSON object. No commentary, no markdown fences. Example:
{"action_type": "plant", "plot_id": 0, "seed_type": "wheat"}"""


def _synthetic_narrative_text(seed: int) -> str:
    """Placeholder day-0 narrative until cell 5's FarmEnvClient is wired post-WS2.
    Variation by seed gives GRPO a non-degenerate prompt distribution."""
    climates = ["temperate", "arid", "tropical"]
    climate = climates[seed % 3]
    rain_chance = {"temperate": "40%", "arid": "10%", "tropical": "70%"}[climate]
    return (
        f"=== Day 1 — {climate.capitalize()} climate ===\n"
        f"Weather: {climate}, rain chance ~{rain_chance} today.\n"
        f"Wallet: $200.00 | Water tank: 50L | Aquifer: 500L | Labor: 10.0h\n"
        f"Plots: 4 empty plots ready for seeds.\n"
        f"Storage: empty.\n"
        f"Market: wheat $8.00, rice $14.00, corn $20.00 (base prices).\n"
        f"\nDecide your first action."
    )


def _build_prompt(seed: int) -> str:
    narrative = _synthetic_narrative_text(seed)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"### Observation:\n{narrative}\n\n### Thought:"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


prompt_dataset = Dataset.from_list([
    {"prompt": _build_prompt(s), "noise_seed": s, "task_id": 1}
    for s in range(NUM_PROMPTS)
])

print(f"prompt_dataset: {len(prompt_dataset)} rows")
print(f"sample prompt[0] head:\n{prompt_dataset[0]['prompt'][:500]}\n... ({len(prompt_dataset[0]['prompt'])} chars total)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9 (per §20.7) — GRPOConfig
# Hyperparameters locked per §20.2. Do NOT edit values without justification:
#   - lr=5e-6 (NOT 5e-5; 5e-5 too aggressive for 0.5B GRPO).
#   - beta=0.0 (no reference model — saves ~30% VRAM/speed).
#   - loss_type="dapo" + epsilon_high=0.28 + delta=1.5 — DAPO two-sided
#     clipping per §20.5 (single biggest stability win on small models).
#   - mask_truncated_completions=True per §20.6 pitfall #3.
#   - max_steps=50 GRPO updates (NOT env iterations); 1 step = 1 batch of K=4.
# ─────────────────────────────────────────────────────────────────────────────
from trl import GRPOConfig

training_args = GRPOConfig(
    # Generation / rollout
    temperature=1.0,                       # GRPO needs exploration
    max_prompt_length=1024,                # narrative_text + system prompt
    max_completion_length=80,              # JSON action only
    num_generations=4,                     # K — group size for advantage estimation
    # Optimizer
    learning_rate=5e-6,
    optim="adamw_8bit",                    # falls back to "adamw_torch" if bnb mismatch (§20.6 #5)
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=0.1,                     # tight clipping for small-model stability
    # Batch / steps
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,         # smoother gradient signal
    max_steps=50,                          # 50 GRPO updates (not 50 env iters)
    # GRPO loss
    beta=0.0,                              # no KL ref model (set 0.04 only if drift)
    epsilon=0.2,
    epsilon_high=0.28,                     # DAPO upper clip
    delta=1.5,                             # two-sided clipping per INTELLECT-2 (§20.5)
    loss_type="dapo",                      # > vanilla GRPO on small models
    mask_truncated_completions=True,       # §20.6 pitfall #3
    importance_sampling_level="token",
    # Logging / checkpointing
    logging_steps=1,
    save_steps=10,
    output_dir="grpo_farm_qwen_0_5b",
    report_to="wandb",
)

print("GRPOConfig ready.")
print(f"  K={training_args.num_generations}  steps={training_args.max_steps}  lr={training_args.learning_rate}")
print(f"  loss_type={training_args.loss_type}  ε={training_args.epsilon}/{training_args.epsilon_high}  δ={training_args.delta}  β={training_args.beta}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 10 (per §20.7) — GRPOTrainer scaffold
# This cell is intentionally NOT runnable end-to-end yet. The reward_funcs
# list references functions defined in cell 7, which lands post-HANDOFF #2
# (A's WS2 economy hardening) and HANDOFF #3 (A's WS3 RubricComposer).
#
# Today this cell only verifies that the wiring is correct: GRPOTrainer
# instantiates against (model, tokenizer, prompt_dataset, training_args).
# The reward_funcs slot is gated by REWARD_FUNCS_READY = False until cell 7
# is written; flip it to True after HANDOFF #2/#3 to actually train.
#
# Per §20.6 pitfall #6: TRL >=0.11 renamed `tokenizer=` to `processing_class=`.
# ─────────────────────────────────────────────────────────────────────────────
from trl import GRPOTrainer

REWARD_FUNCS_READY = False                 # flip to True after cell 7 lands

if REWARD_FUNCS_READY:
    # Cell 7 will define these (post-HANDOFF #2/#3):
    reward_funcs = [
        task_completion_reward,            # Execution anchor (§21 hybrid multi-objective)
        format_reward,                     # JSON-parseable format guide
        heuristic_similarity_reward,       # Edit-distance vs HeuristicAgent
        anti_exploit_reward,               # Penalize known exploit patterns
        stewardship_reward,                # Multi-objective: soil health
    ]

    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,        # NOT tokenizer= (§20.6 pitfall #6)
        reward_funcs=reward_funcs,
        args=training_args,
        train_dataset=prompt_dataset,
    )

    # Uncomment to actually train (1.5–4h on L4/T4):
    # trainer.train()
    print("✅ GRPOTrainer instantiated; call trainer.train() to start.")
else:
    print("⏸️  Cell 10 scaffold — waiting for cell 5 (FarmEnvClient) and cell 7 (reward functions).")
    print("    Set REWARD_FUNCS_READY=True after HANDOFF #2/#3 lands those cells.")
    print("    Sanity check: prompt_dataset =", len(prompt_dataset), "rows; training_args ready.")